# Basics of ITensors

As far as we are concerned, a tensor is a multi-dimensional table. Each element a rank-$N$ tensor is accessed by providing an assignment to the $N$ (integer) indices.

In this course we will use the `ITensors` library (Julia fashion) to work with tensors. The documentation of the library can be found [here](https://docs.itensor.org/ITensors/stable/). In this notebook we will explore some basics of `ITensors`. 

In [1]:
using Pkg
Pkg.activate("..")
using LinearAlgebra
using ITensors

  Activating project at `~/Work/Workspace/Unimi/Corsi/TN_Ulm/Materiale_dispense/TN_Notebooks`


## Indices

Tensors as multi-indexed objects. The most fundamental entity we have to introduce is thus the `Index`.

In [2]:
#Index is a type that represents the indices of tensors in ITensors.jl. It is used to specify the dimensions and properties of the indices when creating tensors. For example, you can create an index with a specific dimension and name like this:

i=Index(2)


(dim=2|id=538)

We see that an index is assigned an `id`. We do not need to remember the `id`, which is something useful to the library. 

Indices can be assigned tags. A tag is a string, and can be used to add useful information.

In [3]:
j=Index(2;tags="TLS")

(dim=2|id=883|"TLS")

You can extract the tags of an index

In [4]:
tags(j)

"TLS"

The tags of an index can be overwritten by

In [5]:
j=settags(j, "BL,") #Mind the comma, it is needed!


(dim=2|id=883|"BL")

An index can have one or more tags, forming the `TagSet`.

In [6]:
j = addtags(j, "TLS,")

(dim=2|id=883|"BL,TLS")

Tags can be used to extract the elements of an array of index that have the tag

In [7]:
appo = [i,j]

2-element Vector{Index{Int64}}:
 (dim=2|id=538)
 (dim=2|id=883|"BL,TLS")

In [8]:
appo[hastags.(appo, "TLS")]

1-element Vector{Index{Int64}}:
 (dim=2|id=883|"BL,TLS")

There are other properties of indices, such as prime level, which we will discuss further on.

## Tensors

Tensors can be constructed starting from a set of indices. Lets take the two indices `i` and `j` defined on the previous part, characterize them both as the indices of a rank-$1$ tensor providing the state of two Spin 1/2 particles.

In [9]:
i=settags(i,"Spin 1/2,")
j=settags(j,"Spin 1/2,")    

(dim=2|id=883|"Spin1/2")

To actually define the tensors we need to construct one by starting from its indices and the type of the values in the tensor.

In [10]:
ψ1 =ITensor(ComplexF64,i)

ITensor ord=1 (dim=2|id=538|"Spin1/2")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}

In [11]:
@show ψ1

ψ1 = ITensor ord=1
Dim 1: (dim=2|id=538|"Spin1/2")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}
 2-element




ITensor ord=1 (dim=2|id=538|"Spin1/2")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}

The tensor $\psi1$ is, so far, empty: the components of the state are yet unspecified. We can assign values componentwise as follows

In [12]:
# state |+> = (|0> + |1>)/sqrt(2)
ψ1[1] = 1/sqrt(2)
ψ1[2] = 1/sqrt(2)
@show ψ1


ψ1 = ITensor ord=1
Dim 1: (dim=2|id=538|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 0.7071067811865475 + 0.0im
 0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=538|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Alternatively we can set the components direcly by assigning the values in an array

In [13]:
stateMinus = ComplexF64[1/sqrt(2), -1/sqrt(2)]
ψ2 = ITensor(ComplexF64,stateMinus, j)
#or simply
#ψ2 = ITensor(ComplexF64, j)

ITensor ord=1 (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We can use the indices `i` and `j` to build an operator acting on a qubit

In [14]:
opMat3 = ComplexF64.([1. 0.;0. -1.])
pauli3 = ITensor(ComplexF64, opMat3, i, j)

ITensor ord=2 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [15]:
@show pauli3

pauli3 = ITensor ord=2
Dim 1: (dim=2|id=538|"Spin1/2")
Dim 2: (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 1.0 + 0.0im   0.0 + 0.0im
 0.0 + 0.0im  -1.0 + 0.0im



ITensor ord=2 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Let's see what happens if we contract the operator with the state ψ1:

In [16]:
out  = pauli3 * ψ1
@show out

out = ITensor ord=1
Dim 1: (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

"It's a kind of magic": the `*` operation as implemented the contraction bewteen the rank-$2$ tensor (operator) `pauli3` and the rank-$1$ tensor $\psi 1$, namely

$
\sum_{i,j=1}^2 M_{i,j}  \psi1_i  \to \phi_j \quad j=1,2
$

It did this because it recognized that there were repeated indices (the `i` index in the state and the same `i` index in the operator) and inferred a contraction over the indices.

As tou can see, the result is a rank-$1$ tensor indexed by `j` (compare the `id`).

Interesting but...wrong...or better: the result is correct: $\sigma_z \ket{+} = \ket{-}$ but this happened by chance...because `pauli3` is symmetric.

We can inspect this issue by using this custom operator

In [17]:
myOp = ITensor(ComplexF64,ComplexF64.([1 2; 3 4]), i, j)

ITensor ord=2 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [18]:
@show myOp * ψ1

myOp * ψ1 = ITensor ord=1
Dim 1: (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  2.82842712474619 + 0.0im
 4.242640687119285 + 0.0im



ITensor ord=1 (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

whereas the matrix-vector multiplication we aimed to was:

In [19]:
ComplexF64.([1 2; 3 4]) * [1/sqrt(2), 1/sqrt(2)]

2-element Vector{ComplexF64}:
 2.1213203435596424 + 0.0im
  4.949747468305832 + 0.0im

So: `ITensor` is indeed be able to perform contractions, but we need to be careful. 

Let's take a step back. 

The index `i` is the _physical_ index of a spin 1/2 particle. It indexes the elements of a basis of an Hilbert space isomorphic to $\mathbb{C}^2$.

The index `j` plays the same role but refers (morally) to the Hilbert space of another system. 

In practice, we must see the states $\psi1$ and $\psi2$ defined by means of two different indices, as objects living in two different Hilbert spaces $\mathcal{H}_1$ and $\mathcal{H}_2$

The correct way do define operators onto $\mathcal{H}_1$ that are able to correctly act on states in the same space is:

$O = \sum_{i'=1}^{2} \sum_{i=1}^2 O_{i',i}\ket{i'}\bra{i}$

In [20]:
pauli3OK = ITensor(ComplexF64, ComplexF64.([1. 0.;0. -1.]), i',i)

ITensor ord=2 (dim=2|id=538|"Spin1/2")' (dim=2|id=538|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [21]:
inds(pauli3OK)

((dim=2|id=538|"Spin1/2")', (dim=2|id=538|"Spin1/2"))

__Note:__ the use of the index `i` to index the columns and of `i'`to index the rows. The "prime" in `i'` primes the index: the stored index is the same, but it has an additional property , the prime level.
The prime level of an index can be inspected thgrough the `plev` accessor function.

In [22]:
plev.([i,i'])

2-element Vector{Int64}:
 0
 1

The application of the operator `pauli3OK` to the state $\psi1$ returns 

The proper way to define an operator on the Hilbert space of the a TLS is:

In [23]:
ϕ1  = pauli3OK * ψ1
@show ϕ1

ϕ1 = ITensor ord=1
Dim 1: (dim=2|id=538|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=538|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

If we want to have the output state with the same (unprimed) index `i` we need to tell `ITensors`

In [24]:
ϕ1  = noprime(pauli3OK * ψ1)
@show ϕ1

ϕ1 = ITensor ord=1
Dim 1: (dim=2|id=538|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=538|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

At last: if we want to extract the "table" of the tensor we can use

In [25]:
matrix(pauli3OK, i', i)

2×2 Matrix{ComplexF64}:
 1.0+0.0im   0.0+0.0im
 0.0+0.0im  -1.0+0.0im

In [26]:
@show matrix(myOp, i, j)
@show matrix(myOp, j, i)

matrix(myOp, i, j) = ComplexF64[1.0 + 0.0im 2.0 + 0.0im; 3.0 + 0.0im 4.0 + 0.0im]
matrix(myOp, j, i) = ComplexF64[1.0 + 0.0im 3.0 + 0.0im; 2.0 + 0.0im 4.0 + 0.0im]


2×2 Matrix{ComplexF64}:
 1.0+0.0im  3.0+0.0im
 2.0+0.0im  4.0+0.0im

### Exercise

For the sake of completeness, let us define the Pauli matrices $\sigma_x,\sigma_y$ and $\sigma_z$ onto the spaces $\mathcal{H}_1$ and $\mathcal{H}_2$.

In [27]:
pauliMatX = ComplexF64.([0. 1.;1. 0.])
pauliMatY = ComplexF64.([0. -im; im 0.])
pauliMatZ = ComplexF64.([1. 0.;0. -1.])
pauliMatId = ComplexF64.([1. 0.;0. 1.])

2×2 Matrix{ComplexF64}:
 1.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im

In [28]:
pauliMats = [pauliMatX, pauliMatY, pauliMatZ, pauliMatId]
# In H_1 (index i)
pauliOps1 = [ITensor(ComplexF64, pauliMat, i', i) for pauliMat in pauliMats]
# In H_2 (index j)
pauliOps2 = [ITensor(ComplexF64, pauliMat, j', j) for pauliMat in pauliMats]

#tuple(array...) converts an array into a tuple, which allows us to unpack the elements of the array into separate variables. In this case, we are unpacking the 4 Pauli operators for each index into separate variables for easier access and readability.

σx1,σy1,σz1,id1 = tuple(pauliOps1...)
σx2,σy2,σz2,id2 = tuple(pauliOps2...)



(ITensor ord=2
Dim 1: (dim=2|id=883|"Spin1/2")'
Dim 2: (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.0 + 0.0im  1.0 + 0.0im
 1.0 + 0.0im  0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=2|id=883|"Spin1/2")'
Dim 2: (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.0 + 0.0im  0.0 - 1.0im
 0.0 + 1.0im  0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=2|id=883|"Spin1/2")'
Dim 2: (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 1.0 + 0.0im   0.0 + 0.0im
 0.0 + 0.0im  -1.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=2|id=883|"Spin1/2")'
Dim 2: (dim=2|id=883|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 1.0 + 0.0im  0.0 + 0.0im
 0.0 + 0.0im  1.0 + 0.0im
)

## Some operations

### Norm

The _norm_ of a state can be computed either as

In [29]:
dag(ψ1) * ψ1

ITensor ord=0
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We obtain a rank-$0$ tensor, i.e. a scalar. We can extract the value:

In [30]:
scalar(dag(ψ1) * ψ1)

0.9999999999999998 + 0.0im

Even simpler:

In [31]:
norm(ψ1)

0.9999999999999999

### Inner product

In [32]:
η = ITensor(ComplexF64, ComplexF64.([1,0]), i)

ITensor ord=1 (dim=2|id=538|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

$\braket{\eta|\psi1}$

In [33]:
scalar(dag(η) * ψ1)

0.7071067811865475 + 0.0im

It should be clear that the order of the arguments does not matter: the contraction occurs over repeated indices

In [34]:
scalar(ψ1 * dag(η))

0.7071067811865475 + 0.0im

### Expectation value

In [35]:
scalar(prime(dag(ψ1)) * σx1 * ψ1)

0.9999999999999998 + 0.0im

or

In [36]:
inner(ψ1', σx1, ψ1)

0.9999999999999998 + 0.0im

### Outer product

If we want to build the density matrix corresponding to the state $\psi1$, i.e.

$ \rho = \ket{\psi1} \bra{\psi1}$

In [37]:
@show ψ1 * prime(dag(ψ1))

ψ1 * prime(dag(ψ1)) = ITensor ord=2
Dim 1: (dim=2|id=538|"Spin1/2")
Dim 2: (dim=2|id=538|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.4999999999999999 + 0.0im  0.4999999999999999 + 0.0im
 0.4999999999999999 + 0.0im  0.4999999999999999 + 0.0im



ITensor ord=2 (dim=2|id=538|"Spin1/2") (dim=2|id=538|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

The `dag` provides a "view" on the state that conjugates the elements, the `prime` avoids unwanted contractions.

### Trace

$ \rho = \ket{\psi1} \bra{\psi1}$

In [38]:
ρ1 = ψ1 * prime(dag(ψ1))
matrix(ρ1, i, i')

2×2 Matrix{ComplexF64}:
 0.5+0.0im  0.5+0.0im
 0.5+0.0im  0.5+0.0im

The trace can be obtained by means of a delta tensor

In [39]:
scalar(ρ1 * delta(i, i'))

0.9999999999999998 + 0.0im

## Composite systems

Let's introduce yet another index which, for what we said above, will refer to another Hilbert space $\mathcal{H_3}$; the system is, this time, a 5 level system (e.g. a boson truncated to the levels 0:4)

In [40]:
k = Index(5;tags="Boson, n=0:4")

(dim=5|id=488|"Boson,n=0:4")

We define the (truncated) bosonic operators

In [41]:
Diagonal([1,2,3])

3×3 Diagonal{Int64, Vector{Int64}}:
 1  ⋅  ⋅
 ⋅  2  ⋅
 ⋅  ⋅  3

In [42]:
aTruncMat = Tridiagonal(zeros(ComplexF64, 4), zeros(ComplexF64, 5), sqrt.(1:4))
aDagTruncMat = aTruncMat' #or adjoint(aTruncMat)
nMat = Diagonal(0:4)


5×5 Diagonal{Int64, UnitRange{Int64}}:
 0  ⋅  ⋅  ⋅  ⋅
 ⋅  1  ⋅  ⋅  ⋅
 ⋅  ⋅  2  ⋅  ⋅
 ⋅  ⋅  ⋅  3  ⋅
 ⋅  ⋅  ⋅  ⋅  4

In [43]:
aDagTruncMat * aTruncMat

5×5 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  2.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  3.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  4.0+0.0im

In [44]:
Diagonal(ComplexF64.(ones(5)))

5×5 Diagonal{ComplexF64, Vector{ComplexF64}}:
 1.0+0.0im      ⋅          ⋅          ⋅          ⋅    
     ⋅      1.0+0.0im      ⋅          ⋅          ⋅    
     ⋅          ⋅      1.0+0.0im      ⋅          ⋅    
     ⋅          ⋅          ⋅      1.0+0.0im      ⋅    
     ⋅          ⋅          ⋅          ⋅      1.0+0.0im

In [45]:
bosMat = [aTruncMat, aDagTruncMat, nMat,Diagonal(ComplexF64.(ones(5)))]
aOp3,aDagOp3,nOp3, id3 = tuple([ITensor(ComplexF64, mat, k', k) for mat in bosMat]...)

(ITensor ord=2
Dim 1: (dim=5|id=488|"Boson,n=0:4")'
Dim 2: (dim=5|id=488|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Base.ReshapedArray{ComplexF64, 1, Tridiagonal{ComplexF64, Vector{ComplexF64}}, Tuple{Base.MultiplicativeInverses.SignedMultiplicativeInverse{Int64}}}}
 5×5
 0.0 + 0.0im  1.0 + 0.0im                 0.0 + 0.0im  …  0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im  1.4142135623730951 + 0.0im     0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im                 0.0 + 0.0im     0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im                 0.0 + 0.0im     2.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im                 0.0 + 0.0im     0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=5|id=488|"Boson,n=0:4")'
Dim 2: (dim=5|id=488|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 5×5
 0.0 - 0.0im                 0.0 - 0.0im  …  0.0 - 0.0im  0.0 - 0.0im
 1.0 - 0.0im                 0.0 - 0.0im     0.0 - 0.0im  0.0 - 0.0im
 0.0 - 0.0im  1.4142135623730951 - 0.0im     0.0 - 0.0im  0.0 - 0.0im
 0.0 - 0.0im              

We now have:
- the indices for all Hilbert spaces: `i`, `j` and `k`
- the operators defined in the corresponding Hilbert spaces

In order to define a state in joined Hilbert space we need a rank-$3$ tensor:

In [144]:
sysState = ITensor(ComplexF64, i,j,k)


ITensor ord=3 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}

We can initialize each system into an arbitrary initial state

In [145]:
#State |+>_1
vector(ψ1)

2-element Vector{ComplexF64}:
 0.7071067811865475 + 0.0im
 0.7071067811865475 + 0.0im

In [146]:
#State |->_2
vector(ψ2)

2-element Vector{ComplexF64}:
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im

In [147]:
#Oscillator state |0>_3 = (1,0,0,0,0),  i.e. vacuum state
ψ3 = ITensor(ComplexF64, ComplexF64.([1,0,0,0,0]), k) 

ITensor ord=1 (dim=5|id=488|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [148]:
for a in 1:2, b in 1:2, c in 1:5
    sysState[i=>a, j=>b, k=>c] = ψ1[i=>a] * ψ2[j=>b] * ψ3[k=>c]
end

Norm of the resulting (product) state

In [149]:
scalar(sysState*dag(sysState))

0.9999999999999996 + 0.0im

For example,  if we want the expectation value of $\sigma^x_1 \otimes \mathbb{1}_2 \otimes \mathbb{1}_3$

In [150]:
inner(sysState',id3*id2*σx1,sysState)

0.9999999999999996 + 0.0im

or

In [151]:
 scalar(noprime(σx1 * sysState)*dag(sysState))

0.9999999999999996 + 0.0im

To compute $\bra{\psi} \mathbb{1}_1 \otimes \sigma^x_2   \otimes \mathbb{1}_3 \ket{\psi}$

In [152]:
inner(sysState',id3*id1*σx2,sysState)

-0.9999999999999996 + 0.0im

If we want the average occupation number of the oscillator:

In [153]:
 scalar(noprime(nOp3 * sysState)*dag(sysState))

0.0 + 0.0im

### Partial Trace

We can obtain the density matrix of the composite system as:

In [154]:
ρsys = sysState * prime(dag(sysState))


ITensor ord=6 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")' (dim=5|id=488|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We can obtain the trace of the density matrix as

In [155]:
scalar(ρsys* delta(i, i')*delta(j, j')*delta(k, k'))

0.9999999999999996 + 0.0im

If we want the partial trace over, for example, the bosonic degree of freedom:

In [156]:
partialRes = ρsys * delta(k, k')

ITensor ord=4 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

The resulting density matrix is equal to $\rho_1 \otimes \rho_2$, since the state is factorized.

In [157]:
sysStateReduced = ITensor(ComplexF64, i, j)
for a in 1:2, b in 1:2
    sysStateReduced[i=>a, j=>b] = ψ1[i=>a] * ψ2[j=>b]
end

In [158]:
ρRed = sysStateReduced * prime(dag(sysStateReduced))

ITensor ord=4 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In order to verify the identity we turn the density matrices $\rho$`Red` and `partialRes` into matrices. To this end it is expedient to use `combiner` objects: combiners, as their name suggests, combine different indices into one single index.

In [159]:
c = combiner(i,j)
cprime = combiner(prime(i), prime(j))

ITensor ord=3 (dim=4|id=37|"CMB,Link") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")'
NDTensors.Combiner

In [160]:
ρRedComb =c * ρRed * cprime 
partialResComb = c * partialRes * cprime

ITensor ord=2 (dim=4|id=479|"CMB,Link") (dim=4|id=37|"CMB,Link")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [161]:
matrix(ρRedComb) == matrix(partialResComb)

true

### Example

Let us consider the above defined system (2 spins 1/2 and a boson). This time we want to:

1. evolve the state of the system under the action of the Hamiltonian:
$$
H = \frac{\mathbb{1} +\sigma_2^z}{2} (a_3 + a_3^\dag)
$$ 
for a time $\delta t = 0.2$, i.e. 
$$
\rho' = e^{-i \delta t H} \; \rho \; e^{i \delta t H};
$$

2. Take the partial trace over the oscillator degree of freedom.

The Hamiltonian is well understood: the oscillator is acted upon by $X \equiv (a_3 +a_3^\dagger)$ only if the spin 2 has non-vanishing amplitude on the state $\ket{\uparrow}$. We can rewrite $H$ as:

$
H = \ket{\uparrow}_2\bra{\uparrow} X_3
$

with $\ket{\uparrow}_2\bra{\uparrow}$ the projector on the eigenstate of $\sigma_2^z$ belonging to the eigenvalue  $+1$ and $X_3 = (a_3 +a_3^\dagger)$.

We define the operators $\ket{\uparrow}_2\bra{\uparrow}$ and $X_3$:

In [162]:
pUp2 = 0.5 * (id2+ σz2);
XOp3 = aOp3 + aDagOp3;

In [163]:
matrix(pUp2,j',j)

2×2 Matrix{ComplexF64}:
 1.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im

In [164]:
matrix(XOp3, k', k)

5×5 Matrix{ComplexF64}:
 0.0+0.0im      1.0+0.0im      0.0+0.0im      0.0+0.0im  0.0+0.0im
 1.0+0.0im      0.0+0.0im  1.41421+0.0im      0.0+0.0im  0.0+0.0im
 0.0+0.0im  1.41421+0.0im      0.0+0.0im  1.73205+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im  1.73205+0.0im      0.0+0.0im  2.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im      2.0+0.0im  0.0+0.0im

We can then define the Hamiltonian:

In [165]:
ham = id1 * pUp2 * XOp3

ITensor ord=6 (dim=2|id=538|"Spin1/2")' (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2")' (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4")' (dim=5|id=488|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

and, given $\delta t = 0.2$, compute:

In [166]:
δt = 0.2
uEv = exp(-1im * δt * ham)

ITensor ord=6 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")' (dim=5|id=488|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

OK...a bit too fast: we exponentiated a rank-$6$ tensor! Well, believe this or not this is exactly the kind of features that a good library for tensor handling should offer. After all, we know that we can always turn a tensor into a matrix and that, if we do the reshaping in a physically meaningful way, we will end up with a matrix. Given that the operator is the tensor product of Hermitian operators, the resulting matrix will be Hermitian as well etc...

But, in case you don't trust ITensor, we can proceed with a check. First of all, since $H$ acts trivially on $\mathcal{H}_1$ we can restrict our attention to:

In [167]:
#Trace over index i
#and mind that the trace of the identity is the dimension of the index, which in this case is 2, hence the factor 0.5
redHam = ham * 0.5 *delta(i, i')
#Convert to rank-2 tensor via combiners
#mind the order of the indices in the combiner, which is the opposite of the order in the Hamiltonian, because of how the multiplication works
c2 = combiner(k,j,tags="combined j,k")
c3 = combiner(k',j',tags="combined j',k'")
redHamMatr = c2 * redHam * c3
matrix(redHamMatr)


10×10 Matrix{ComplexF64}:
 0.0+0.0im      1.0+0.0im      0.0+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 1.0+0.0im      0.0+0.0im  1.41421+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  1.41421+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im  1.73205+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im

We can compare the resulting operator with the tensor product of $\ket{\uparrow}_2\bra{\uparrow}\otimes X_3$ 

In [168]:
kron(matrix(pUp2, j', j),matrix(XOp3, k', k))

10×10 Matrix{ComplexF64}:
 0.0+0.0im      1.0+0.0im      0.0+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 1.0+0.0im      0.0+0.0im  1.41421+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  1.41421+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im  1.73205+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im      0.0+0.0im      0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im

We can do the same comparison with the exponentiated Hamiltonian:

In [169]:
#Trace over index i
#and mind that the trace of the identity is the dimension of the index, which in this case is 2, hence the factor 0.5
reduEv = uEv * 0.5 *delta(i, i')
#Convert to rank-2 tensor via combiners
#mind the order of the indices in the combiner, which is the opposite of the order in the Hamiltonian, because of how the multiplication works
c2 = combiner(k,j,tags="combined j,k")
c3 = combiner(k',j',tags="combined j',k'")
reduEvMatr = c2 * reduEv * c3
matrix(reduEvMatr)

10×10 Matrix{ComplexF64}:
   0.980199+0.0im         …  0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0-0.19604im        0.0+0.0im  0.0+0.0im  0.0+0.0im
 -0.0277242+0.0im            0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.00320119im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.00032227+0.0im            0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im         …  0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im            0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im            1.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im            0.0+0.0im  1.0+0.0im  0.0+0.0im
        0.0+0.0im            0.0+0.0im  0.0+0.0im  1.0+0.0im

In [170]:
exp(-im *δt * matrix(redHamMatr))

10×10 Matrix{ComplexF64}:
   0.980199+0.0im         …  0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0-0.19604im        0.0+0.0im  0.0+0.0im  0.0+0.0im
 -0.0277242+0.0im            0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.00320119im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.00032227+0.0im            0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im         …  0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im            0.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im            1.0+0.0im  0.0+0.0im  0.0+0.0im
        0.0+0.0im            0.0+0.0im  1.0+0.0im  0.0+0.0im
        0.0+0.0im            0.0+0.0im  0.0+0.0im  1.0+0.0im

In [171]:
matrix(reduEvMatr) ≈ exp(-im *δt * matrix(redHamMatr))

true

...ok...I hope you trust ITensor (which, internally, does something similar to what we deed, in an optimized way).

The implementation of the evolution of $\rho_S$ via ITensor constructs is a little bit involved. As per definition, the state tensor $\rho$`sys` is:

$\rho_S = \sum_{i,j,k, i',j',k'} \rho_{i,j,k}^{i',j',k'} \ket{i,j,k} \bra{i',j',k'}$

In [172]:
ρsys

ITensor ord=6 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")' (dim=5|id=488|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We must therefore pay attention: if we contracted $\rho_S$ with $O = e^{-i \delta t H}$ we would get a rank-$0$ tensor, since __all__ of the indices would be contracted. What we want is something different. Even worse, we need to act on both the left side and the right side of $\rho_S$. 

In [173]:
uEv*ρsys

ITensor ord=0
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

and...

In [174]:
uEv*ρsys*dag(uEv)

ITensor ord=6 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")' (dim=5|id=488|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [175]:
scalar(uEv*ρsys)

0.9900993366550546 + 0.0im

...is even worse since it is indeed something like $\lambda * O^\dagger$, which is not what we want, but has the right rank!

In oder to accomplish what we want we need to be cunning.

1. Introduce a new set of indices:

In [207]:
#the function sim creates a new index that is a "simulated" version of the original index, meaning that it has the same dimension and properties but is treated as a different index in tensor operations. 
i1 = noprime(sim(i',tags="simulated_i'"))
j1 = noprime(sim(j',tags="simulated_j'"))
k1 = noprime(sim(k',tags="simulated_k'"))
#The noprime is used to remove the prime from the simulated index, so that it can be used in tensor operations without being treated as a different index. This allows us to create a new tensor that has the same properties as the original tensor but is treated as a different tensor in operations.

(dim=5|id=698|"simulated_k'")

We use this to modify the indices of the operator $O$:
$$
O_{i,j,k}^{i',j',k'} \to O_{i,j,k}^{i_1,j_1,k_1}
$$

In [208]:
appouEv = uEv *delta(i', i1) * delta(j', j1) * delta(k', k1)

ITensor ord=6 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4") (dim=2|id=986|"simulated_i'") (dim=2|id=880|"simulated_j'") (dim=5|id=698|"simulated_k'")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We can now contract $\rho$ with $O$

In [209]:
appouEv *ρsys

ITensor ord=6 (dim=2|id=986|"simulated_i'") (dim=2|id=880|"simulated_j'") (dim=5|id=698|"simulated_k'") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")' (dim=5|id=488|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We can now multiply by $O^\dagger$. This operation is safe since
$$
O^\dagger = \sum_{i,j,k,i',j',k'} 
\left( O_{i,j,k}^{i',j',k'} \right )^*
\ket{i,j,k} \bra{i',j',k'} 
\stackrel{\text{swap idx} \leftrightarrow \text{idx'}}{=}
\sum_{i,j,k,i',j',k'} 
\left( O_{i',j',k'}^{i,j,k} \right )^*\ket{i',j',k'} \bra{i,j,k}
$$
so that

In [210]:
newρ = appouEv *ρsys*dag(uEv)

ITensor ord=6 (dim=2|id=986|"simulated_i'") (dim=2|id=880|"simulated_j'") (dim=5|id=698|"simulated_k'") (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

is the right result. However, we end up with a tensor 
$$\rho(\delta t) = 
\sum_{i_1,j_1,k_1i,j,k} 
O_{i_1,j_1,k_1}^{i,j,k}
\ket{i_1,j_1,k_1} \bra{i,j,k}
$$
while we started form 
$$\rho = 
\sum_{i_1,j_1,k_1,i,j,k} 
O_{i,j,k}^{i',j',k'} \ket{i,j,k} \bra{i',j',k'}
$$

We restore the original form by means, as usual, of $\delta$ tensors:

In [190]:
step1 = newρ * delta(i,i')*delta(j,j')*delta(k,k')
step2 = step1 * delta(i1, i)*delta(j1, j)*delta(k1, k)
rhoδt = step2

ITensor ord=6 (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")' (dim=5|id=488|"Boson,n=0:4")' (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

After all this ado we have the result. Mind: we are doing all of this as to make you aware that, while automated contraction over repeated indices can be a most useful tool, it has its downsides and the user must always be aware of them. The tensor notation presented in the lecture is the way to go to __design__ tensor operations and allows to spot possible criticalities.

### Checking the result

We started from a pure state and evolved it by means of a unitary operator. The resulting state is therefore pure. We could have obtained the result in a much simpler way. As a matter of fact:
$$
\rho(\delta t) = \ket{\psi(\delta t)}\bra{\psi(\delta t)}
$$
with
$$
\ket{\psi(\delta t)} = e^{-i \delta t H} \ket{\psi}
$$

In [222]:
appoPsi = noprime(uEv * sysState) 
toCheck = appoPsi * dag(prime(appoPsi))

ITensor ord=6 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4") (dim=2|id=538|"Spin1/2")' (dim=2|id=883|"Spin1/2")' (dim=5|id=488|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Let's have a look at the reduced density matrix of the sybsystems 2 and 3 as obtained by evolving the density matrix

In [223]:
matrix(rhoδt* delta(i, i')*combiner(k, j) * combiner(prime(k), prime(j)))

10×10 Matrix{ComplexF64}:
    0.480395+0.0im        …  0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0-0.0960789im     0.0+0.0im  0.0+0.0im  0.0+0.0im
  -0.0135876+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0015689im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.000157944+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
   -0.490099+0.0im        …  0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im

Let's compare it with the reduced density matrix of subsystems 2 and 3 as obtained by the evolved state

In [224]:
matrix(toCheck* delta(i, i')*combiner(k, j) * combiner(prime(k), prime(j)))

10×10 Matrix{ComplexF64}:
    0.480395+0.0im        …  0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0-0.0960789im     0.0+0.0im  0.0+0.0im  0.0+0.0im
  -0.0135876+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0015689im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.000157944+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
   -0.490099+0.0im        …  0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im
         0.0+0.0im           0.0+0.0im  0.0+0.0im  0.0+0.0im

In [225]:
toCheck ≈ rhoδt

true

Apart from numerical differences, the two reduced stated do match! Well done!

## Schmidt decomposition

Let's take the  initial and evolved states:

In [228]:
ψ = sysState
ψEv = appoPsi

ITensor ord=3 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=5|id=488|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Consider now a bipartition of the system $(1,2,3) \to (1,2) + (3)$. Since the initial state is factorized, there is no entanglement across any partition, and in particular across $(1,2),(3)$. We can test this by reshaping the state $\ket{\psi}$ into a rank-$2$ tensor and performing a Schmidt decomposition of the resulting matrix.

### Intial (factorized) state

In [254]:
#ITensor SVD is a function that performs singular value decomposition (SVD) on a given tensor. It takes the tensor to be decomposed, the indices to be grouped together for the left and right singular vectors. The function returns three tensors: the left singular vectors (uu), the singular values (ss), and the right singular vectors (vv). In this case, we are performing SVD on the tensor ψ, grouping indices i and j together for the left singular vectors.

uu,ss, vv = svd(ψ, (i, j))

ITensors.TruncSVD(ITensor ord=3
Dim 1: (dim=2|id=538|"Spin1/2")
Dim 2: (dim=2|id=883|"Spin1/2")
Dim 3: (dim=4|id=6|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2×4
[:, :, 1] =
                0.5 + 0.0im  -0.5000000000000001 + 0.0im
 0.5000000000000001 + 0.0im  -0.5000000000000001 + 0.0im

[:, :, 2] =
                0.0 + 0.0im   0.7886751345948129 + 0.0im
 0.5773502691896257 + 0.0im  -0.2113248654051871 + 0.0im

[:, :, 3] =
                0.0 + 0.0im  -0.2113248654051871 + 0.0im
 0.5773502691896257 + 0.0im   0.7886751345948129 + 0.0im

[:, :, 4] =
   0.8660254037844387 + 0.0im  0.28867513459481287 + 0.0im
 -0.28867513459481287 + 0.0im  0.28867513459481287 + 0.0im
, ITensor ord=2
Dim 1: (dim=4|id=6|"Link,u")
Dim 2: (dim=4|id=825|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}
 4×4
 0.9999999999999998   0.0   0.0  0.0
 0.0                 -0.0   0.0  0.0
 0.0                  0.0  -0.0  0.0
 0.0                  0.0   0.0  0.0
, ITensor ord=2
Dim 1: (dim=5|id=488|"B

In [255]:
# Rank-3 tensor: two physical indices and a bond index.
uu

ITensor ord=3 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=4|id=6|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [256]:
#Rank-2 tensor: singular values. Two bond indices: one from the left singular vectors and one from the right singular vectors.
ss

ITensor ord=2 (dim=4|id=6|"Link,u") (dim=4|id=825|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}

In [260]:
#inspect ss
matrix(ss)

4×4 Matrix{Float64}:
 1.0   0.0   0.0  0.0
 0.0  -0.0   0.0  0.0
 0.0   0.0  -0.0  0.0
 0.0   0.0   0.0  0.0

Notice that the matrix `ss` is $4 \times 4$: its side length equals the size of $\text{min}(2^2,5)$, where $2^2$ and $5$ are the physical dimensions of the spaces $\mathcal{H}_1 \otimes \mathcal{H}_2$ and $\mathcal{H}_3$ respectively.

In [259]:
#Here rank-2 tensor: one bond index (the same as the )
vv

ITensor ord=2 (dim=5|id=488|"Boson,n=0:4") (dim=4|id=825|"Link,v")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

The SVD decomposition is correct.

In [247]:
uu*ss*dag(vv) ≈ ψ

true

In [262]:
@show uu-dag(uu)

uu - dag(uu) = ITensor ord=3
Dim 1: (dim=2|id=538|"Spin1/2")
Dim 2: (dim=2|id=883|"Spin1/2")
Dim 3: (dim=4|id=6|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2×4
[:, :, 1] =
 0.0 + 0.0im  0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im

[:, :, 2] =
 0.0 + 0.0im  0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im

[:, :, 3] =
 0.0 + 0.0im  0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im

[:, :, 4] =
 0.0 + 0.0im  0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im



ITensor ord=3 (dim=2|id=538|"Spin1/2") (dim=2|id=883|"Spin1/2") (dim=4|id=6|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

The entanglement entropy

$$
S= - \sum_{j} s_j^2 * \log(s_j^2)
$$

is therefore zero.

`uu` is unitary and the "columns" of `uu` are the left singular vectors:

In [270]:
uuReshaped = combiner(i, j,tags="(i,j)") * uu 
uuMatrix = matrix(uuReshaped)

4×4 Matrix{ComplexF64}:
  0.5+0.0im        0.0+0.0im        0.0+0.0im   0.866025+0.0im
  0.5+0.0im    0.57735+0.0im    0.57735+0.0im  -0.288675+0.0im
 -0.5+0.0im   0.788675+0.0im  -0.211325+0.0im   0.288675+0.0im
 -0.5+0.0im  -0.211325+0.0im   0.788675+0.0im   0.288675+0.0im

In [281]:
[round.(uuMatrix * (uuMatrix)') ≈ Diagonal(ComplexF64.(ones(4))),round.(uuMatrix' * (uuMatrix)) ≈ Diagonal(ComplexF64.(ones(4)))]


2-element Vector{Bool}:
 1
 1

`vv` is unitary and the rows of `vv`$^\dagger$ are the right singular vectors

In [306]:
i1, i2 = inds(vv)
vvAppo = prime(vv, i1)
vv * dag(vvAppo)
vvMatrix = matrix(vv)

5×4 Matrix{ComplexF64}:
 1.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  1.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  1.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im

5×5 Matrix{ComplexF64}:
 1.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0-0.0im  1.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0-0.0im  0.0-0.0im  1.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  1.0+0.0im  0.0+0.0im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0+0.0im

### Evolved state

For the evolved state:

In [245]:
uuEv, ssEv, vvEv = svd(ψEv, (i, j))

ITensors.TruncSVD(ITensor ord=3
Dim 1: (dim=2|id=538|"Spin1/2")
Dim 2: (dim=2|id=883|"Spin1/2")
Dim 3: (dim=4|id=97|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2×4
[:, :, 1] =
 0.5 + 0.0im                    …  -0.49999999999999983 - 4.717328049479396e-19im
 0.5 - 9.247837391336543e-19im     -0.49999999999999983 - 4.717328049479396e-19im

[:, :, 2] =
 0.49999999999999994 + 0.0im                    …  0.5000000000000001 + 4.717485293506616e-17im
  0.4999999999999999 + 9.248145652110278e-17im                    0.5 + 4.7174852935066175e-17im

[:, :, 3] =
  0.7071067811865475 + 0.0im                    …  -2.8776230423487246e-16 - 3.010946563721465e-17im
 -0.7071067811865477 - 6.474034418700819e-17im        4.53046031526745e-17 - 3.5938720255938477e-17im

[:, :, 4] =
  1.198032307611922e-16 + 0.0im                     …   0.7068974912807944 - 0.017202814331355305im
 -7.018811786099334e-17 - 5.6311396309316895e-19im     -0.7068974912807944 + 0.017202814331355305im
, ITenso

The squares of the singular values, which corresponds to the norm, is

In [253]:
sum(diag(matrix(ssEv)) .^2) ≈ 1

true

In [248]:
function entEntropy(svMatrix::Matrix{<:Number})
    s = 0.0
    for sv in diag(svMatrix)
        if sv > 1e-10 # Avoid log(0) issues
            s -= sv^2 * log(sv^2)
        end
    end
    return s
end

entEntropy (generic function with 1 method)

In [249]:
entEntropy(matrix(ssEv))

0.05554457050536878

..."small" but non-zero...as expected.

Same considerations done for `uu`, `ss` and `vv` apply to `uuEv`, `ssEv`and `vvEv`.